# Technology

Time-based One-Time Password (TOTP) authentication is commonly known for the qrcodes you scan with your phone and the six digit temporary codes the authenticator generates based on this qrcode. Here is such a qrcode:

![qrcode](images/qrcode.png)

There are three distinct steps to an authenticator like `pyauthenticator`:
* Decoding the qrcode to access the included information.
* Generate an access code based on this information.
* Finally, for completeness generating a qrcode.

This notebook walks through all three steps individually, using low-level calls to the underlying libraries, before `pyauthenticator` wraps them into a single convenient interface.

## Libraries
Rather than reinventing the wheel, we are going to use three libraries, which already provide the required functionality:
* [pyotp](https://github.com/pyauth/pyotp) - Python One-Time Password Library
* [pyzbar](https://github.com/NaturalHistoryMuseum/pyzbar/) - Read one-dimensional barcodes and QR codes from Python 2 and 3.
* [qrcode](https://github.com/lincolnloop/python-qrcode) - Python QR Code image generator

## Decoding

A qrcode used for authenticator apps encodes a single `otpauth://` URI as a string. `pyzbar` reads the raw bytes out of the image; no TOTP-specific logic is involved yet, it is purely image decoding.

In [1]:
from PIL import Image
from pyzbar.pyzbar import decode

In [ ]:
qrcode_png_file_name = "images/qrcode.png"
result = decode(Image.open(qrcode_png_file_name))[0].data.decode("utf-8")

In [3]:
result

'otpauth://totp/Looker%20Authentication?secret=JBYFSYBPBA2SSCM2'

`result` is the decoded `otpauth://` URI. It contains the service name, the issuer, and a `secret` query parameter, which is the actual shared secret the TOTP algorithm needs. `pyauthenticator` stores this URI as-is; the secret is only extracted from it right before a code is generated.

In [4]:
secret = result.split("=")[-1]
secret

'JBYFSYBPBA2SSCM2'

## Generate Access Code

`pyotp.TOTP` implements [RFC 6238](https://datatracker.ietf.org/doc/html/rfc6238): it combines the shared `secret` with the current time to derive a short numeric code. `.now()` returns the code valid for the current time step (30 seconds by default), the same code an authenticator app would display right now.

In [5]:
import pyotp

In [9]:
pyotp.TOTP(s=secret).now()

'426192'

### Under the hood: implementing TOTP from RFC 6238

`pyotp.TOTP(...).now()` hides five steps defined by [RFC 6238](https://datatracker.ietf.org/doc/html/rfc6238), which extends the counter-based [HOTP algorithm (RFC 4226)](https://datatracker.ietf.org/doc/html/rfc4226) by deriving the counter from time instead of an incrementing counter:

1. **Shared secret** — the Base32 `secret` decoded above; known to both the authenticator app and the service, and never transmitted again after the initial qrcode scan.
2. **Time counter** — the current Unix time is divided into fixed windows, 30 seconds by default: `T = floor((unix_time − T0) / X)`, with `T0 = 0` and step `X = 30`.
3. **HMAC** — `T` is packed into an 8-byte big-endian integer and hashed together with the secret key using HMAC-SHA1 (SHA256/SHA512 are also valid, selected via the `otpauth://` URI's `algorithm` parameter), producing a 20-byte digest.
4. **Dynamic truncation** — the low nibble of the digest's *last* byte is used as an offset; the 4 bytes starting at that offset are read as a big-endian integer with the top bit cleared.
5. **Modulo** — that integer, taken modulo `10^digits` (6 by default) and left-padded with zeros, is the code shown on screen.

The cell below reimplements these five steps directly, without `pyotp`, and produces the same code as `pyotp.TOTP(...).now()` above (both are computed for the same 30-second window).

In [10]:
import base64
import hashlib
import hmac
import struct
import time


def totp_from_scratch(secret_b32: str, digits: int = 6, step: int = 30, t0: int = 0) -> str:
    padding = "=" * ((8 - len(secret_b32) % 8) % 8)
    key = base64.b32decode(secret_b32.upper() + padding)
    counter = int((time.time() - t0) // step)
    counter_bytes = struct.pack(">Q", counter)

    digest = hmac.new(key, counter_bytes, hashlib.sha1).digest()
    offset = digest[-1] & 0x0F
    truncated = struct.unpack(">I", digest[offset : offset + 4])[0] & 0x7FFFFFFF

    return str(truncated % (10**digits)).zfill(digits)


totp_from_scratch(secret)

'426192'

## Generate QRcode

Going the other direction, `qrcode` can re-encode the same `otpauth://` URI as an image. This is what powers `pyauthenticator`'s `--qrcode` option: it lets a service already configured in `pyauthenticator` be re-added to a phone's authenticator app without needing the original qrcode again.

In [7]:
import qrcode

In [8]:
qrcode.make(result).save("test.png", "PNG")

## Security considerations

The properties that make TOTP convenient for automation are the same ones worth understanding when reasoning about its security.

**What TOTP protects against well:**
* **Short-lived codes** — each code is only valid for one time-step (30 seconds by default, plus the small drift tolerance below), so a code intercepted after it expires is worthless.
* **No secret in transit** — after the initial qrcode scan, only the 6-digit code crosses the network; the shared secret itself never has to be sent again.
* **Replay resistance** — a server that tracks the last accepted time-step rejects a previously-used code outright, even if it's still within its validity window.
* **Offline generation** — codes are derived locally from the secret and the clock, with no SMS/email delivery step to intercept or redirect (unlike SMS-based two-factor authentication, which is also vulnerable to SIM swapping).

**What TOTP does *not* protect against:**
* **Real-time phishing (adversary-in-the-middle)** — a fake login page that immediately forwards a captured password *and* TOTP code to the real service can still authenticate as the victim, since the code stays valid for several seconds after it's generated. TOTP raises the bar over a password alone, but unlike [FIDO2/WebAuthn passkeys](https://webauthn.guide/), it does not stop this class of attack.
* **Secret/seed compromise** — anyone who obtains the shared secret (from a compromised device, a leaked backup, or a stolen credential file) can generate valid codes indefinitely, without needing to intercept anything in real time.
* **Malware on the trusted device** — if the device holding the secret is itself compromised, malware can read the secret or the generated codes directly.
* **Clock drift** — the counter `T` from the previous section only matches between authenticator and server if their clocks roughly agree. Servers therefore accept a small window of adjacent time-steps (commonly ±1, i.e. ±30 seconds) to tolerate drift; a device whose clock has drifted further than that will generate codes the server rejects.

Because of the secret-compromise point above, using TOTP from an automated process — as `pyauthenticator` is designed to do — changes the traditional security model: the value of two-factor authentication comes from combining *something you know* (a password) with *something you have* (a device only the user controls), and once both live on the same automated machine, that separation is gone. See [pyauthenticator's security considerations](https://github.com/jan-janssen/pyauthenticator#security-considerations) for how to weigh that trade-off, and this [background article on TOTP](https://medium.com/@raditya.mit/totp-demystified-how-time-based-one-time-passwords-secure-your-logins-3798339ed29c) for further reading.

The result is again the same qrcode used in the beginning:

![qrcode](images/qrcode.png)

Together, these three steps — decode, generate, (re-)encode — are exactly what `pyauthenticator._core` implements; the [Implementation](implementation.ipynb) notebook covers that module directly.